In [6]:
from kiteconnect import KiteConnect
import pandas as pd
import numpy as np
import ta
import datetime as dt


In [7]:
api_key="z6mjwdwdm8b1301h"
secret_key="ods20s5dgrrmfp41ml73kgue2fnfnldv"
kite=KiteConnect(api_key=api_key)
print(kite.login_url())

https://kite.zerodha.com/connect/login?api_key=z6mjwdwdm8b1301h&v=3


In [8]:
request_token="siYg4FWeCcmbNXs2r5z1HxkThU8sAi2q"
data=kite.generate_session(request_token=request_token,api_secret=secret_key)
data

{'user_type': 'individual/res_no_nn',
 'email': 'manshu143stocks@gmail.com',
 'user_name': 'Manshu Sharma',
 'user_shortname': 'Manshu',
 'broker': 'ZERODHA',
 'exchanges': ['NSE', 'MF', 'BSE'],
 'products': ['CNC', 'NRML', 'MIS', 'BO', 'CO'],
 'order_types': ['MARKET', 'LIMIT', 'SL', 'SL-M'],
 'avatar_url': None,
 'user_id': 'TNM718',
 'api_key': 'z6mjwdwdm8b1301h',
 'access_token': '1bIxRcnECkePTT39i8GpUK2eBItDS1jo',
 'public_token': '8pOXd0nD2fMG4mfSs0TjKdtJL4GN6dJT',
 'refresh_token': '',
 'enctoken': 'uVva+7O33CPjpJOg6s2CutVLE8G5PGqlOkYomSlnpHkDY/N/E5CHOsmZa3pAZhR4Asog6SrAsEbcmouP+We0PEWjR9bYAXXeKRiViqH6HXiZyrMhgHjgVNdXMiIBb+w=',
 'login_time': datetime.datetime(2025, 4, 21, 9, 17, 13),
 'meta': {'demat_consent': 'consent'}}

In [9]:

# Initialize KiteConnect
kite.set_access_token(data["access_token"])

# List of stock symbols (example)
symbols = ["NSE:INFY", "NSE:TCS", "NSE:HDFCBANK", "NSE:RELIANCE"]

# Time range for historical data
from_date = (dt.datetime.today() - dt.timedelta(days=30)).strftime('%Y-%m-%d')
to_date = dt.datetime.today().strftime('%Y-%m-%d')

results = []

for symbol in symbols:
    try:
        # Fetch historical data
        data = kite.historical_data(
            instrument_token=kite.ltp(symbol)[symbol]['instrument_token'],
            from_date=from_date,
            to_date=to_date,
            interval="day"
        )

        df = pd.DataFrame(data)

        if not df.empty:
            df['returns'] = df['close'].pct_change()
            volatility = df['returns'].std() * np.sqrt(252)  # annualized
            total_return = (df['close'].iloc[-1] / df['close'].iloc[0] - 1) * 100

            # Calculate RSI
            df['rsi'] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()
            latest_rsi = df['rsi'].iloc[-1]

            results.append({
                'symbol': symbol,
                'total_return_%': total_return,
                'volatility': volatility,
                'rsi': latest_rsi
            })

    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")

# Convert to DataFrame
result_df = pd.DataFrame(results)

# Rank stocks based on desired factors
result_df['return_rank'] = result_df['total_return_%'].rank(ascending=False)
result_df['volatility_rank'] = result_df['volatility'].rank()  # lower volatility better
result_df['rsi_rank'] = result_df['rsi'].rank()  # optional, depends on your strategy

# Composite score (example: equal weightage)
result_df['final_score'] = result_df[['return_rank', 'volatility_rank', 'rsi_rank']].mean(axis=1)

# Sort based on final score
ranked_stocks = result_df.sort_values(by='final_score')

print(ranked_stocks[['symbol', 'total_return_%', 'volatility', 'rsi', 'final_score']])


         symbol  total_return_%  volatility        rsi  final_score
1       NSE:TCS       -9.147274    0.221572  26.119371     1.666667
2  NSE:HDFCBANK        7.705556    0.282549  73.357037     2.333333
3  NSE:RELIANCE       -2.227171    0.289257  55.135831     2.666667
0      NSE:INFY       -9.665673    0.329931  35.921759     3.333333
